# Philadelphia CIP – Image-Based PDF Parser (2021–2025)

Parses image-based Philadelphia CIP PDFs (2021–2025) into per-year CSVs.
Skips 2019 and 2020 (text-based, handled separately).

**Dependencies:** `pip install pymupdf pdfplumber pytesseract pillow opencv-python-headless pandas`

Tesseract must be installed and on PATH.
Windows: also set `pytesseract.pytesseract.tesseract_cmd`.

## Cell 1 – Imports & paths

In [ ]:
import re
import warnings
from pathlib import Path

import cv2
import numpy as np
import pdfplumber
import fitz          # pymupdf
import pytesseract
import pandas as pd
from PIL import Image

warnings.filterwarnings('ignore')

# Windows: set this if tesseract is not on PATH
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

SCRIPTS_DIR = Path('.').resolve()
CITY_DIR    = SCRIPTS_DIR.parent / 'Philadelphia'
PDF_DIR     = CITY_DIR / 'PDF'
CSV_DIR     = CITY_DIR / 'CSV'
CSV_DIR.mkdir(exist_ok=True)

SKIP_STEMS = {'2019', '2020'}

REQUIRED_COLS = [
    'cip_year', 'project_type', 'source_page', 'department',
    'project_name', 'project_id', 'address_location',
    'start_year', 'end_year',
    'project_description', 'project_justification',
    'previous_appropriations', 'project_total',
]

SRC_CODES = {
    'CN', 'CT', 'CR', 'CA', 'A',
    'XN', 'XT', 'XR', 'Z',
    'FB', 'FO', 'FT',
    'PB', 'PT',
    'SB', 'SO', 'ST',
    'TB', 'TO', 'TT',
    'GO', 'GR', 'GN', 'GP',
}

YEAR_RE  = re.compile(r'^(\d{4})$')
NUM_RE   = re.compile(r'^-?[\d,]+$')
COMBINED = re.compile(r'^[\d,]+[A-Z]{1,3}$')
PROJ_RE  = re.compile(r'^(\d+[A-Z]?)\.?\s+(.+)$')
DEPT_RE  = re.compile(r'^[A-Z][A-Z &\-/()\u2013]+')

TESS_CFG = '--psm 6 --oem 1'
OCR_DPI  = 300

pdfs_to_process = sorted(
    p for p in PDF_DIR.glob('*.pdf') if p.stem not in SKIP_STEMS
)
print('Paths OK.')
print(f'  PDF_DIR : {PDF_DIR}')
print(f'  CSV_DIR : {CSV_DIR}')
print('PDFs to process:', [p.name for p in pdfs_to_process])

## Cell 2 – Page classification

In [ ]:
def classify_pages(pdf_path, min_page=15, max_text_chars=300):
    candidates = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, pg in enumerate(pdf.pages):
            pg_num = i + 1
            if pg_num < min_page:
                continue
            if len(pg.images) >= 1 and len(pg.chars) < max_text_chars:
                candidates.append(pg_num)
    return candidates


def is_data_page(ocr_text):
    years_found = len(re.findall(r'\b20\d{2}\b', ocr_text))
    has_unit    = bool(re.search(r'x000|\$x|x 000', ocr_text, re.I))
    return years_found >= 4 and has_unit


for p in pdfs_to_process:
    print(f'{p.name}: {len(classify_pages(p))} candidate image pages')

## Cell 3 – Image extraction & OCR

In [ ]:
def ocr_page(pdf_path, pg_1idx, dpi=OCR_DPI):
    doc   = fitz.open(str(pdf_path))
    mat   = fitz.Matrix(dpi / 72, dpi / 72)
    pix   = doc[pg_1idx - 1].get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
    doc.close()
    arr   = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    arr   = clahe.apply(arr)
    _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return pytesseract.image_to_string(Image.fromarray(arr), config=TESS_CFG)


# Uncomment to spot-check OCR quality:
# for p in pdfs_to_process:
#     cands = classify_pages(p)
#     if cands:
#         text = ocr_page(p, cands[0])
#         print(f'\n{p.name}  page {cands[0]}:')
#         for line in text.split('\n')[:10]:
#             if line.strip(): print(' ', repr(line.strip()))

print('OCR helpers ready.')

## Cell 4 – Line classifier

In [ ]:
DEPT_KEYWORDS = {
    'MUSEUM', 'AVIATION', 'COMMERCE', 'FINANCE', 'FIRE', 'FLEET',
    'LIBRARY', 'HEALTH', 'HOMELESS', 'HOUSING', 'SERVICES',
    'LICENSES', 'MANAGING', 'MURAL', 'PARKS', 'PLANNING', 'POLICE',
    'PRISONS', 'PROPERTY', 'RECORDS', 'REVENUE', 'SCHOOLS',
    'SEPTA', 'STREETS', 'TRANSIT', 'WATER', 'ZOO', 'OIT',
    'SUSTAINABILITY', 'MDO', 'OHS',
}

def is_src(tok):
    return tok.upper() in SRC_CODES

def parse_num(s):
    try:
        return float(str(s).replace(',', '').strip())
    except Exception:
        return None

def is_department(text):
    return any(kw in text.upper() for kw in DEPT_KEYWORDS)

def classify_line(line, plan_years):
    line = line.strip()
    if not line or re.match(r'^\d{1,3}$', line):
        return ('skip',)
    if re.search(r'x000|\$x|x 000', line, re.I):
        return ('unit',)
    tokens = line.split()
    yrs = [int(t) for t in tokens if YEAR_RE.match(t) and 2018 <= int(t) <= 2040]
    if len(yrs) >= 4:
        return ('year_hdr', list(dict.fromkeys(yrs)))
    if re.match(r'^TOTALS?\s*[-\u2013]', line, re.I):
        return ('totals_hdr',)
    nb_start = None
    for i, tok in enumerate(tokens):
        if NUM_RE.match(tok.replace(',', '')) or is_src(tok) or COMBINED.match(tok):
            nb_start = i
            break
    if nb_start is not None:
        left_text = ' '.join(tokens[:nb_start]).strip()
        right     = tokens[nb_start:]
        has_sc    = any(is_src(t) or COMBINED.match(t) for t in right)
        nums = []
        for t in right:
            if COMBINED.match(t):
                nums.append(parse_num(re.sub(r'[A-Z]+$', '', t)))
            elif NUM_RE.match(t.replace(',', '')):
                nums.append(parse_num(t))
        nums = [n for n in nums if n is not None]
        if not nums:
            return ('text', line)
        if has_sc:
            return ('src_row', nums, left_text)
        yr_vals = {plan_years[j]: v for j, v in enumerate(nums[:-1]) if j < len(plan_years)}
        range_total = nums[-1] if len(nums) > 1 else (nums[0] if nums else None)
        return ('total_row', yr_vals, range_total, left_text)
    m = PROJ_RE.match(line)
    if m:
        name = m.group(2).strip()
        if not re.search(r'\btotal\b', name, re.I):
            return ('project', m.group(1).rstrip('.'), name)
    if DEPT_RE.match(line) and len(line.replace(' ', '')) > 3 and 'TOTAL' not in line.upper():
        return ('dept', line)
    return ('text', line)

print('Line classifier ready.')

## Cell 5 – Page parser

**Project hierarchy tracking:**

Philadelphia CIPs nest projects: a parent project line (e.g. `74 SEPTA Bridge...`) acts as a title header with no funding rows of its own. Sub-projects (`1`, `2`, `3` ...) follow immediately underneath and carry the actual source/total rows.

The resulting `project_id` format is `DEPT.parent.child` (e.g. `DEPT.74.1`).

Detection: if `pending` has no `pending_yr` and no `has_funding_rows` when the next project line arrives, the pending entry is a header. Its number becomes `main_proj_num`, and sub-project IDs are built as `DEPT.main_proj_num.child_num`.

In [ ]:
def _flush(ctx, projects):
    pending    = ctx.get('pending')
    pending_yr = ctx.get('pending_yr')
    if pending is None:
        return
    if not pending_yr:
        return  # no year values; project may continue on next page
    name = pending.get('project_name', '').strip()
    if name and len(name) >= 3 and name[0] not in ("'", '|', '"', '.', ','):
        for yr, v in pending_yr.items():
            pending[f'year_{yr}'] = v
        rt = ctx.get('pending_tot')
        pending['project_total']         = rt if rt else sum(pending_yr.values())
        pending['project_description']   = ' '.join(ctx.get('desc_buf', [])).strip()
        pending['project_justification'] = pending['project_description']
        projects.append(pending)
    ctx['pending']          = None
    ctx['pending_yr']       = None
    ctx['pending_tot']      = None
    ctx['desc_buf']         = []
    ctx['has_funding_rows'] = False


def _hard_flush(ctx, projects):
    if ctx.get('pending_yr'):
        _flush(ctx, projects)
    else:
        ctx['pending']          = None
        ctx['pending_yr']       = None
        ctx['pending_tot']      = None
        ctx['desc_buf']         = []
        ctx['has_funding_rows'] = False


def _is_header(ctx):
    # A pending project with no funding rows is a parent title line.
    return (
        ctx.get('pending') is not None
        and not ctx.get('pending_yr')
        and not ctx.get('has_funding_rows')
    )


def parse_page(ocr_text, pg_num, cip_year, ctx, projects):
    plan_years = ctx.get('plan_years', [])
    for raw_line in ocr_text.split('\n'):
        cat = classify_line(raw_line, plan_years if plan_years else list(range(2022, 2028)))

        if cat[0] == 'year_hdr':
            new_years = cat[1][:6]
            if new_years != plan_years:
                plan_years = new_years
                ctx['plan_years'] = plan_years

        elif cat[0] in ('unit', 'skip'):
            pass

        elif cat[0] == 'totals_hdr':
            _hard_flush(ctx, projects)
            ctx['main_proj_num'] = ''  # reset hierarchy at section totals

        elif cat[0] == 'dept':
            _hard_flush(ctx, projects)
            txt = cat[1].strip().upper()
            if is_department(txt):
                ctx['dept']          = txt
                ctx['proj_type']     = ''
                ctx['main_proj_num'] = ''  # reset hierarchy on new department
            else:
                ctx['proj_type'] = cat[1].strip()

        elif cat[0] == 'src_row':
            _, nums, left_text = cat
            if ctx.get('pending') is not None:
                ctx['has_funding_rows'] = True
                if left_text:
                    ctx['desc_buf'].append(left_text)

        elif cat[0] == 'total_row':
            _, yr_vals, range_total, left_text = cat
            pending = ctx.get('pending')
            if pending is not None and yr_vals and ctx.get('pending_yr') is None:
                if not (range_total and sum(yr_vals.values()) > range_total * 3):
                    ctx['has_funding_rows'] = True
                    ctx['pending_yr']  = yr_vals
                    ctx['pending_tot'] = range_total
                    _flush(ctx, projects)

        elif cat[0] == 'project':
            _, num_str, name = cat
            dept    = ctx.get('dept', '')
            dept_id = re.sub(r'\s+', '', dept)[:5] if dept else 'UNK'

            # If pending has seen no funding rows, it is a parent header.
            # Record its number as main_proj_num before flushing.
            if _is_header(ctx):
                pid_str    = ctx['pending'].get('project_id', '')
                parent_num = pid_str.rsplit('.', 1)[-1] if '.' in pid_str else pid_str
                ctx['main_proj_num'] = parent_num

            _flush(ctx, projects)

            main = ctx.get('main_proj_num', '')

            # Numeric comparison: incoming > parent -> new top-level project
            # incoming <= parent -> sub-project under that parent
            try:
                num_int  = int(re.sub(r'[A-Za-z]', '', num_str))
                main_int = int(re.sub(r'[A-Za-z]', '', main)) if main else 0
                is_top   = (not main) or (num_int > main_int)
            except (ValueError, TypeError):
                is_top = not main

            if is_top:
                pid = f'{dept_id}.{num_str}'
                ctx['main_proj_num'] = ''  # will be set only if confirmed header
            else:
                pid = f'{dept_id}.{main}.{num_str}'

            ctx['pending'] = {
                'cip_year': cip_year, 'project_type': ctx.get('proj_type', ''),
                'source_page': pg_num, 'department': dept,
                'project_name': name, 'project_id': pid,
                'address_location': '', 'start_year': '', 'end_year': '',
                'project_description': '', 'project_justification': '',
                'previous_appropriations': '', 'project_total': '',
            }
            ctx['has_funding_rows'] = False
            ctx['pending_yr']  = None
            ctx['pending_tot'] = None
            ctx['desc_buf']    = []

        elif cat[0] == 'text':
            if cat[1].strip() and ctx.get('pending') is not None:
                ctx['desc_buf'].append(cat[1].strip())


print('Page parser ready.')

## Cell 6 – PDF-level parser

In [ ]:
def parse_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    stem     = pdf_path.stem
    cip_year = int(stem) if stem.isdigit() else None
    plan_years = []
    with pdfplumber.open(pdf_path) as pdf:
        for pg in pdf.pages[:30]:
            txt = pg.extract_text() or ''
            m   = re.search(
                r'(?:FY|FISCAL YEARS?)\s*(\d{4})\s*[-\u2013]\s*(\d{4})',
                txt, re.I,
            )
            if m:
                y1, y2 = int(m.group(1)), int(m.group(2))
                plan_years = list(range(y1, min(y2 + 1, y1 + 7)))[:6]
                if cip_year is None: cip_year = y1 - 1
                break
    if not plan_years and cip_year:
        plan_years = list(range(cip_year + 1, cip_year + 7))
    print(f'  cip_year={cip_year}  plan_years={plan_years}')
    image_pages = classify_pages(pdf_path)
    total = len(image_pages)
    print(f'  candidate image pages: {total}')
    ctx = {
        'plan_years':       plan_years,
        'dept':             '',
        'proj_type':        '',
        'main_proj_num':    '',   # current parent project number for hierarchy
        'has_funding_rows': False,
        'pending':          None,
        'pending_yr':       None,
        'pending_tot':      None,
        'desc_buf':         [],
    }
    projects = []; data_pages = 0
    for i, pg_num in enumerate(image_pages, 1):
        if i % 20 == 0:
            print(f'  OCR: {i}/{total} (page {pg_num})...', flush=True)
        try:
            ocr_text = ocr_page(pdf_path, pg_num)
        except Exception as exc:
            print(f'  OCR error page {pg_num}: {exc}'); continue
        if not is_data_page(ocr_text): continue
        data_pages += 1
        parse_page(ocr_text, pg_num, cip_year, ctx, projects)
    _flush(ctx, projects)
    print(f'  data pages processed: {data_pages}')
    return cip_year, plan_years, projects


print('PDF parser ready.')

## Cell 7 – DataFrame builder

In [ ]:
def build_dataframe(projects, plan_years):
    if not projects:
        return pd.DataFrame(columns=REQUIRED_COLS)
    year_cols = [f'year_{y}' for y in sorted(plan_years)]
    all_cols  = REQUIRED_COLS + year_cols
    df = pd.DataFrame(projects)
    for col in all_cols:
        if col not in df.columns:
            df[col] = ''
    # Derive start_year / end_year; cast to str to match column dtype
    for idx, row in df.iterrows():
        active = {
            y: row.get(f'year_{y}')
            for y in plan_years
            if pd.notna(row.get(f'year_{y}')) and row.get(f'year_{y}', 0) not in ('', 0, 0.0)
        }
        if active:
            df.at[idx, 'start_year'] = str(min(active))
            df.at[idx, 'end_year']   = str(max(active))
    return df[all_cols].fillna('')

print('DataFrame builder ready.')

## Cell 8 – Main loop: parse all PDFs → write CSVs

In [ ]:
summary = []

for pdf_path in pdfs_to_process:
    print(f'\n== {pdf_path.name} ==')
    cip_year, plan_years, projects = parse_pdf(pdf_path)
    print(f'  rows extracted: {len(projects)}')
    df       = build_dataframe(projects, plan_years)
    csv_path = CSV_DIR / f'{pdf_path.stem}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f'  -> {csv_path}')
    for col in REQUIRED_COLS:
        if col in df.columns:
            null_pct = (df[col].astype(str).str.strip() == '').mean() * 100
            if null_pct > 50:
                print(f'    WARN {col}: {null_pct:.0f}% empty')
    summary.append({
        'pdf': pdf_path.name, 'cip_year': cip_year,
        'plan_years': plan_years, 'rows': len(df), 'csv': str(csv_path),
    })

print('\n== ALL DONE ==')
for s in summary:
    print(f"{s['pdf']}  ->  {s['rows']} rows  plan_years={s['plan_years']}")

## Cell 9 – Output validation

In [ ]:
print('=== Validation Summary ===\n')
for s in summary:
    csv_path = Path(s['csv'])
    if not csv_path.exists():
        print(f'{csv_path.name}: FILE MISSING'); continue
    df = pd.read_csv(csv_path, dtype=str)
    year_cols = [c for c in df.columns if c.startswith('year_')]
    print(f'{csv_path.name}')
    print(f'  Rows         : {len(df)}')
    print(f'  Year columns : {year_cols}')
    print(f'  Depts unique : {df["department"].nunique()}')
    print(f'  Missing name : {(df["project_name"].str.strip() == "").sum()}')
    print(f'  Missing total: {(df["project_total"].astype(str).str.strip() == "").sum()}')
    sample_cols = ['department', 'project_id', 'project_name', 'project_total'] + year_cols[:3]
    with pd.option_context('display.max_colwidth', 40):
        print(df[sample_cols].head(3).to_string(index=False))
    print()